# Build a travel planner agent

You will build one travel assistant. It has three tools. For every question it decides
on its own which tool to use — sometimes one, sometimes two, and sometimes none.

| Tool | What it answers | Who runs it |
|---|---|---|
| `search_flights` | flights between two cities on one date | **your code** |
| `get_city_weather` | weather now, and the next three days | **the platform** |
| `convert_currency` | today's exchange rate | **the platform** |

Nothing in your code says "now call the weather tool". The model decides. It decides from
three things only: the tool names, the tool descriptions, and the system prompt. All three
live in Acrux Core, so you can change how the agent behaves without shipping new code.

Here is the order of work:

1. put the three tool definitions in the catalog,
2. put the system prompt in the catalog,
3. connect the tools to the prompt,
4. run the agent and watch which tools it picks,
5. break it on purpose four times, so you know what each error looks like.

Every cell runs against a real account. Nothing here is faked or mocked.

**Two ways to do every step.** Each step that creates something has two headings:
**In the dashboard**, with a screenshot, and **The same thing in code**, with a cell to run.
They are not two different features — the dashboard and the SDK call the same API, so the
result is identical.

Pick whichever you prefer. Click through the dashboard and skip the code cell, or run the
code cell and never open the dashboard. You do not need both, and doing both is harmless:
every code cell looks for what already exists before it creates anything.

**Three kinds of code cell.** Most of this notebook is not the agent. Before you copy
anything into a project of your own, check which kind of cell you are looking at — each one
is labelled:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or explains what just happened | no |

Only three or four cells are marked **Your app**, and the last step of this notebook lists
exactly what they add up to. Anything else — including the `find_tool` and
`create_tool_if_missing` helpers below — exists so the notebook can be re-run safely and so
you can see what happened. Neither is part of the SDK, and nothing here requires them.

**Companion tutorial:** [Build a travel planner agent](https://docs.acruxcore.com/docs/tutorials/build-a-travel-planner-agent)

**Read this first if tools are new to you:**
[Define a tool in code, or in the catalog](https://docs.acruxcore.com/docs/guides/define-a-tool-in-code-or-in-the-catalog)
— it explains the one idea this notebook builds on: a tool has a *definition* and an
*implementation*, and they do not have to live in the same place.

---

## Step 0 — What you need before you start

If your account is new, do these four things first. Each one takes about a minute.

**1. Create an account** at [acruxcore.com](https://acruxcore.com) and verify your email.

**2. Make a personal API key.** In the dashboard go to **Account & keys → New key**. Copy
the value now — it is shown once and stored hashed. If you lose it, make another one.

**3. Connect a model.** A new account cannot reach any LLM yet. This is the step most
people skip. Two parts, both in the dashboard:

- **Gateway → Credentials → New credential** — paste a provider key (OpenAI, OpenRouter,
  or another provider).
- **Gateway → Models → New model** — give it a **public name** and point it at that
  credential. The public name is the string you pass as the model in code. This notebook
  uses `gpt-4o-mini`. If yours has another name, change `MODEL` in the next cell.

Step by step with screenshots:
[Route your app's LLM calls through the gateway](https://docs.acruxcore.com/docs/guides/route-calls-through-the-gateway).

**4. Install the SDK.**

In [ ]:
# client_tools, used in step 6, landed after 0.9.0 - so upgrade rather than install.
%pip install -q --upgrade acruxcore

Now set your key and the base URL.

**Setup.** Use real environment variables if you can. The two `os.environ` lines below are here so
the notebook runs on its own, but remember that a key typed into a notebook also gets
saved inside the notebook file.

In [1]:
import json
import os

# Better: set these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")

MODEL = "gpt-4o-mini"          # a public name from Gateway -> Models
PROMPT = "travel-planner"      # the prompt this notebook creates

FLIGHTS = "search_flights"       # your code runs this one
WEATHER = "get_city_weather"     # the platform runs this one
CURRENCY = "convert_currency"    # the platform runs this one too

### Preflight

**Check.** Run this before anything else. It checks the four things above, in the order they
usually fail. You get one clear message instead of a long stack trace later.

In [2]:
import inspect

import httpx

import acruxcore
from acruxcore import AcruxCore
from acruxcore.gateway_api import GatewayNamespace

hub = AcruxCore()          # reads ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL

# 1. Does the key work at all?
await hub.prompts.list(limit=1)
print("api key: ok")

# 2. Does this SDK version know client_tools? Step 9 needs it.
has_client_tools = "client_tools" in inspect.signature(
    GatewayNamespace.run_prompt_with_tools
).parameters
print("client_tools supported:", has_client_tools)
if not has_client_tools:
    print(f"  !! acruxcore {acruxcore.__version__} is too old - upgrade it")

# 3. Is there a model to run on, and is MODEL one of them?
#    There is no SDK method for this yet, so call the endpoint directly.
async with httpx.AsyncClient() as http:
    res = await http.get(
        f"{os.environ['ACRUXCORE_BASE_URL']}/gateway/models",
        headers={"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}"},
    )
models = [m["publicName"] for m in res.json()]
print("models on this team:", models or "NONE - add one in Gateway -> Models")
print(f"MODEL {MODEL!r} available:", MODEL in models)

api key: ok
client_tools supported: True
models on this team: ['gpt-4o-mini']
MODEL 'gpt-4o-mini' available: True


---

## Step 1 — How a tool is stored

Read this step slowly. Almost every confusion about tools comes from here.

### A tool in the catalog is two objects, not one

| Object | What it holds | How many |
|---|---|---|
| the **shell** | the tool's `name`, plus a description for your team | one per tool |
| a **version** | the `description` the model reads, the parameter schema, and the executor | many, and each one is frozen |

So creating a tool is **two calls**, not one. The first call makes the shell. The second
call commits version 1.

**A shell on its own cannot be called.** Its page in the dashboard says "No versions yet",
and the model never sees it. This is the single most common surprise: the tool looks
created, and nothing happens.

Why two objects? Because a version is **immutable** — frozen the moment you commit it. To
change a schema you commit a new version, and the old one stays exactly as it was. Every
trace can then say which version really ran. The shell just keeps the name steady while
versions come and go.

### Which description does the model read?

Both objects have a field called `description`, and this is the second common surprise.
The rule is one line of server code:

```
what the model reads  =  version.description  or, if that is empty,  tool.description
```

So the **version's** description is the one that matters. Write it for the model: say
what the tool does and when to use it. The shell's description is only a fallback.

### The executor: who actually runs the tool

A version also names its **executor**. This is the field that decides whose machine runs
the work:

| Executor | Who calls the real API | What the platform stores |
|---|---|---|
| `client` | your own code | the definition only — never your code or your data |
| `http` | the platform, server-side | the URL, the headers, and any secret to send |

Our travel planner needs both kinds:

| Tool | Executor | Why |
|---|---|---|
| `search_flights` | `client` | flight inventory sits in your own database |
| `get_city_weather` | `http` | a public weather API — nothing private to hide |
| `convert_currency` | `http` | a public exchange-rate API |

With that, the next steps are just typing.

---

## Step 2 — Create the `search_flights` shell

This is the tool people find confusing, so the next four steps build it slowly and check the
result after every part. This step makes only the shell: the name, and nothing else.

### In the dashboard

**Gateway → Tools → New tool.** Only a name and an optional description. Nothing about
parameters yet.

| Field | What to enter |
|---|---|
| **Name** | `search_flights` |
| **Description** | Search available flights between two cities on one date. |

![The New tool dialog with the name field set to search_flights and a one-line description](../../../../apps/docs/static/img/tutorials/build-a-travel-planner-agent/14-notebook-new-tool-shell.png)

### The same thing in code

**Setup.** The cell below makes the same shell over the API. It looks for the tool first, so running
this notebook twice does not create a duplicate — and if you already made the tool by
clicking, it prints "already exists" and creates nothing. Pick one route; you do not need
both.

In [3]:
async def find_tool(name: str):
    """Return the catalog tool with exactly this name, or None.

    A notebook helper, not an SDK function. It only exists so re-running this notebook
    does not create duplicates. The one real call inside it is hub.tools.list(), and
    the filter is there because `search` matches substrings, not exact names.
    """
    matches = [t for t in (await hub.tools.list(search=name)).data if t.name == name]
    return matches[0] if matches else None


flights_tool = await find_tool(FLIGHTS)
if flights_tool is None:
    flights_tool = await hub.tools.create(
        FLIGHTS,
        description="Search available flights between two cities on one date.",
    )
    print("created the shell:", flights_tool.name)
else:
    print("the shell already exists:", flights_tool.name)

print("tool id:", flights_tool.id)
print("versions it has so far:", (await hub.tools.list_versions(flights_tool.id)).total)

the shell already exists: search_flights
tool id: 677f23d9-a30f-4860-9e04-5881d8a38919
versions it has so far: 2


If that printed `versions it has so far: 0`, the tool exists but is not usable yet. That
is the shell-with-no-version state from step 1.

## Step 3 — Commit version 1

The shell has no version yet, so nothing can call it. This step fixes that.

Three fields carry the whole definition, whichever route you take.

**Description** — what the model reads on every single call. The dialog says so under the
field.

**Parameters** — one row per argument the model may send: a name, a type, a description,
and a required checkbox. Together these rows are a JSON Schema.

**Executor** — leave it on **Client**, which means "the calling app runs this one".

### In the dashboard

**New version** on the tool's page, then enter this:

| Field | What to enter |
|---|---|
| **Description** | Search available flights between two cities on one date, from the agency's own inventory. Returns the airline, flight number, departure and arrival times, and the price in euros. |
| **Changelog** | leave it empty — it is a note for your team, never sent to the model |
| **Executor** | Client — the caller's app runs it |

Then **Add parameter** three times, one row each:

| Name | Type | Description | req |
|---|---|---|---|
| `origin` | string | Departure city name, e.g. 'Amsterdam'. | ✓ |
| `destination` | string | Arrival city name, e.g. 'Lisbon'. | ✓ |
| `departure_date` | string | Departure date as YYYY-MM-DD. | ✓ |

![The New version dialog for search_flights: a description, three parameter rows for origin, destination and departure date, and the executor set to Client](../../../../apps/docs/static/img/tutorials/build-a-travel-planner-agent/15-notebook-new-version-client.png)

### The same thing in code

**Setup.** Read the schema carefully. The model can only send the fields you list, and it must send
everything in `required`. Each parameter row in the dialog above is one entry in
`properties` here — same three fields, written out instead of typed in.

In [4]:
FLIGHTS_SCHEMA = {
    "type": "object",
    "properties": {
        "origin": {"type": "string", "description": "Departure city name, e.g. 'Amsterdam'."},
        "destination": {"type": "string", "description": "Arrival city name, e.g. 'Lisbon'."},
        "departure_date": {"type": "string", "description": "Departure date as YYYY-MM-DD."},
    },
    "required": ["origin", "destination", "departure_date"],
}

# Written for the model, not for your team. It says what the tool does and what comes back.
FLIGHTS_DESCRIPTION = (
    "Search available flights between two cities on one date, from the agency's own "
    "inventory. Returns the airline, flight number, departure and arrival times, and "
    "the price in euros."
)

if (await hub.tools.list_versions(flights_tool.id)).total == 0:
    version = await hub.tools.commit_version(
        flights_tool.id,
        parameters_schema=FLIGHTS_SCHEMA,
        executor={"type": "client"},        # "the calling app runs this one"
        description=FLIGHTS_DESCRIPTION,
    )
    print(f"committed version {version.version_number}")
    print("aliases minted with it:", version.aliases)
else:
    print("this tool already has a version - nothing committed")

this tool already has a version - nothing committed


When this really is the **first** version, the output also shows two aliases. Every tool's
first version automatically gets `production` and `staging`, both pointing at it. An alias
is a moving label: point `production` at a newer version later, and everything that follows
that alias picks up the change with no redeploy. Later versions do not move any alias for
you — you move it yourself with `promote_alias`.

You can see both aliases on the tool's **Aliases** tab in the dashboard. `Promote` is how
you move one to another version later:

![The Aliases tab of the search_flights tool: production and staging both pointing at v2, each with a version dropdown and a Promote button](../../../../apps/docs/static/img/tutorials/build-a-travel-planner-agent/16-notebook-tool-aliases.png)

## Step 4 — Check what the model will actually read

**Check.** Do not trust the dashboard here — ask the platform for the exact object it will
send to the model. `tools.resolve` follows an alias and returns the resolved definition.

In [5]:
resolved_flights = (await hub.tools.resolve([{"name": FLIGHTS, "alias": "production"}]))[0]

print("version:", resolved_flights.version_number)
print("executor:", resolved_flights.executor_type, " <- 'client' means your code runs it")
print()
print("this is the whole object the model sees:")
print(json.dumps(resolved_flights.function, indent=2))

version: 2
executor: client  <- 'client' means your code runs it

this is the whole object the model sees:
{
  "name": "search_flights",
  "description": "Search available flights between two cities on one date, from the agency's own inventory. Returns the airline, flight number, departure and arrival times, and the price in euros.",
  "parameters": {
    "type": "object",
    "required": [
      "origin",
      "destination",
      "departure_date"
    ],
    "properties": {
      "origin": {
        "type": "string",
        "description": "Departure city name, e.g. 'Amsterdam'."
      },
      "destination": {
        "type": "string",
        "description": "Arrival city name, e.g. 'Lisbon'."
      },
      "departure_date": {
        "type": "string",
        "description": "Departure date as YYYY-MM-DD."
      }
    }
  }
}


That JSON is the entire prompt the model gets about this tool. Three names, three
descriptions, one `required` list. If the model calls the tool with the wrong arguments,
or never calls it, this text is what you fix — not your Python.

## Step 5 — Write the implementation

**Your app.** This one really does ship. The platform stores no code and no data for a
`client` tool, so the function is yours to write. One rule matters:

**The function's parameter names must be the schema's field names.** The loop calls your
function with keyword arguments taken straight from the model's JSON, like
`search_flights(origin=..., destination=..., departure_date=...)`. It does not pass one
dictionary. If a schema field collides with a Python keyword — `from` on a currency tool,
for example — accept `**kwargs` instead.

In [6]:
# Stands in for the flight inventory your own app already has. Keys are
# "origin|destination", lowercased. The standalone scripts beside this notebook read the
# same rows from ../data/flights.json.
INVENTORY = {
    "amsterdam|lisbon": [
        {"flight_no": "KL1693", "airline": "KLM", "depart": "07:20", "arrive": "09:45", "price_eur": 189},
        {"flight_no": "TP671", "airline": "TAP Air Portugal", "depart": "12:05", "arrive": "14:30", "price_eur": 154},
        {"flight_no": "HV5171", "airline": "Transavia", "depart": "18:40", "arrive": "21:05", "price_eur": 118},
    ],
    "amsterdam|tokyo": [
        {"flight_no": "KL861", "airline": "KLM", "depart": "14:35", "arrive": "08:55", "price_eur": 742},
        {"flight_no": "NH218", "airline": "ANA", "depart": "19:10", "arrive": "13:40", "price_eur": 815},
    ],
}


def search_flights(origin: str, destination: str, departure_date: str) -> dict:
    """Look up flights in the in-house inventory.

    The parameter names copy the schema's field names on purpose - that is how the tool
    loop calls this function.
    """
    print(f"  -> search_flights(origin={origin!r}, destination={destination!r}, departure_date={departure_date!r})")
    rows = INVENTORY.get(f"{origin.strip().lower()}|{destination.strip().lower()}", [])
    return {
        "origin": origin,
        "destination": destination,
        "departure_date": departure_date,
        "flights": rows,
        "count": len(rows),
    }


# It is an ordinary function. Nothing about the platform is involved in calling it.
print(search_flights("Amsterdam", "Lisbon", "2026-08-28")["count"], "flights found")

  -> search_flights(origin='Amsterdam', destination='Lisbon', departure_date='2026-08-28')
3 flights found


---

## Step 6 — Create the two tools the platform runs

`get_city_weather` and `convert_currency` are built the same way — shell, then version —
with one difference: their executor is `http`, so the platform makes the call and your app
does nothing at run time.

An `http` executor is a small recipe for one HTTP request: a method, a URL, query
parameters, headers. The model's arguments are filled into it with `{{arg.<name>}}`
placeholders. Any secret it needs is stored on the platform and never reaches your code.

One extra field is worth knowing about: **`responseTransform`**. The weather API answers
with about 39 KB of JSON. All of it would be sent back to the model as input tokens on the
next round, and you pay for every one. A `responseTransform` is a small JavaScript
function that runs on the server and keeps only the useful fields, which brings the result
under 500 bytes.

### In the dashboard

Exactly like `search_flights` — **New tool**, then **New version** — with the executor
dropdown switched from **Client** to **HTTP**. That reveals the method, URL, header and
query fields:

![The version dialog with the executor set to HTTP, method GET, and a URL ending in a templated city argument, with hints that a value may reference a stored secret or a model argument](../../../../apps/docs/static/img/tutorials/build-a-travel-planner-agent/03-http-executor-fields.png)

Here is everything to enter, so you do not have to read it out of the code below.

**Tool 1 — `get_city_weather`**

| Field | What to enter |
|---|---|
| **Name** (shell) | `get_city_weather` |
| **Description** (shell) | Current conditions and a three-day forecast for one city. |
| **Description** (version) | Get the current weather and a three-day forecast for one city. The forecast only reaches three days ahead, so it cannot answer a question about a season or a typical climate. |
| **Parameter** | name `city`, type `string`, description *City name, e.g. 'Lisbon' or 'Kyoto'. English names work best.*, required ✓ |
| **Executor** | HTTP — the gateway calls a URL |
| **Method** | `GET` |
| **URL** | `https://wttr.in/{{arg.city}}` |
| **Query** | one row: name `format`, value `j1` |
| **Response transform** | the `WEATHER_TRANSFORM` text from the next cell — copy everything between the triple quotes |

**Tool 2 — `convert_currency`**

| Field | What to enter |
|---|---|
| **Name** (shell) | `convert_currency` |
| **Description** (shell) | Convert an amount from one currency to another at today's rate. |
| **Description** (version) | Convert an amount of money from one currency to another at today's reference exchange rate. |
| **Parameters** | `amount` · number · *How much to convert, e.g. 500.* · ✓<br/>`from` · string · *Three-letter ISO code of the source currency, e.g. 'EUR'.* · ✓<br/>`to` · string · *Three-letter ISO code of the target currency, e.g. 'JPY'.* · ✓ |
| **Executor** | HTTP — the gateway calls a URL |
| **Method** | `GET` |
| **URL** | `https://api.frankfurter.dev/v1/latest` |
| **Query** | three rows: `amount` = `{{arg.amount}}`, `base` = `{{arg.from}}`, `symbols` = `{{arg.to}}` |
| **Response transform** | leave empty — this API's answer is already small |

Note how each `{{arg.…}}` placeholder spells a parameter name exactly. That is the whole
link between what the model sends and where it lands in the request. A typo there is not
caught for you: the placeholder is simply left unfilled.

### The same thing in code

**Setup.** `create_tool_if_missing` below is a helper this notebook defines - not an SDK
function. It wraps the same two real calls you made by hand for `search_flights`,
`hub.tools.create()` and `hub.tools.commit_version()`, and skips whichever already exists
so you can run the cell twice. Every value from the tables above appears here too, so this
is also the cell to copy the transform from.

In [7]:
WEATHER_TRANSFORM = """function transform(input) {
  var b = input.body || {};
  var cur = (b.current_condition || [])[0] || {};
  var area = (b.nearest_area || [])[0] || {};
  var pick = function (list) {
    var first = (list || [])[0] || {};
    return first.value || null;
  };
  var days = (b.weather || []).map(function (d) {
    var hours = d.hourly || [];
    var noon = hours.filter(function (h) { return h.time === '1200'; })[0] || {};
    return {
      date: d.date,
      max_c: Number(d.maxtempC),
      min_c: Number(d.mintempC),
      midday_conditions: pick(noon.weatherDesc),
      chance_of_rain_pct: noon.chanceofrain === undefined ? null : Number(noon.chanceofrain)
    };
  });
  return {
    city: pick(area.areaName),
    country: pick(area.country),
    current: {
      temp_c: Number(cur.temp_C),
      feels_like_c: Number(cur.FeelsLikeC),
      conditions: pick(cur.weatherDesc),
      humidity_pct: Number(cur.humidity)
    },
    forecast: days
  };
}"""


async def create_tool_if_missing(name: str, *, shell_description: str, description: str, schema: dict, executor: dict):
    """Create the shell, then commit version 1 - but only where they are missing.

    A notebook helper, NOT an SDK function. There is no `ensure` anything in the API: this
    wraps the same two real calls you made by hand for search_flights, hub.tools.create()
    and hub.tools.commit_version(), and skips whichever already exists so the notebook can
    be re-run. In your own project you would just make the two calls once, or click twice
    in the dashboard.
    """
    tool = await find_tool(name)
    if tool is None:
        tool = await hub.tools.create(name, description=shell_description)
        print(f"{name}: created the shell")
    if (await hub.tools.list_versions(tool.id)).total == 0:
        version = await hub.tools.commit_version(
            tool.id, parameters_schema=schema, executor=executor, description=description
        )
        print(f"{name}: committed version {version.version_number}")
    else:
        print(f"{name}: already has a version, left as it is")
    return tool


weather_tool = await create_tool_if_missing(
    WEATHER,
    shell_description="Current conditions and a three-day forecast for one city.",
    description=(
        "Get the current weather and a three-day forecast for one city. The forecast only "
        "reaches three days ahead, so it cannot answer a question about a season or a "
        "typical climate."
    ),
    schema={
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name, e.g. 'Lisbon' or 'Kyoto'. English names work best.",
            }
        },
        "required": ["city"],
    },
    executor={
        "type": "http",
        "method": "GET",
        "url": "https://wttr.in/{{arg.city}}",     # the model's `city` goes here
        "query": [{"name": "format", "value": "j1"}],
        "headers": [],
        "argMapping": [],
        "responseTransform": WEATHER_TRANSFORM,
    },
)

get_city_weather: already has a version, left as it is


**Setup.** The second one has three arguments, and all three go into the query string instead
of the path. Look at how each `{{arg.…}}` placeholder matches a schema field name.

In [8]:
currency_tool = await create_tool_if_missing(
    CURRENCY,
    shell_description="Convert an amount from one currency to another at today's rate.",
    description=(
        "Convert an amount of money from one currency to another at today's reference "
        "exchange rate."
    ),
    schema={
        "type": "object",
        "properties": {
            "amount": {"type": "number", "description": "How much to convert, e.g. 500."},
            "from": {
                "type": "string",
                "description": "Three-letter ISO code of the source currency, e.g. 'EUR'.",
            },
            "to": {
                "type": "string",
                "description": "Three-letter ISO code of the target currency, e.g. 'JPY'.",
            },
        },
        "required": ["amount", "from", "to"],
    },
    executor={
        "type": "http",
        "method": "GET",
        "url": "https://api.frankfurter.dev/v1/latest",
        "query": [
            {"name": "amount", "value": "{{arg.amount}}"},
            {"name": "base", "value": "{{arg.from}}"},
            {"name": "symbols", "value": "{{arg.to}}"},
        ],
        "headers": [],
        "argMapping": [],
    },
)

convert_currency: already has a version, left as it is


### Check that the platform really can run it

**Check.** You do not have to wait for the agent to prove this. Ask the platform to execute
the tool right now, with arguments you choose. This is also the fastest way to debug an `http`
executor: if it fails here, it will fail in the agent too.

In [9]:
weather_out = await hub.tools.execute(weather_tool.id, {"city": "Lisbon"})

print("http status:", weather_out.status, " time taken:", weather_out.latency_ms, "ms")
print("version that ran:", weather_out.tool_version_id)
print()
print(json.dumps(weather_out.result, indent=2))

http status: 200  time taken: 778 ms
version that ran: 247f6c1c-8762-49f1-b63b-b2ba0a5e2b1c

{
  "city": "Lisbon",
  "country": "Portugal",
  "current": {
    "temp_c": 25,
    "feels_like_c": 20,
    "conditions": "Sunny",
    "humidity_pct": 45
  },
  "forecast": [
    {
      "date": "2026-08-21",
      "max_c": 25,
      "min_c": 18,
      "midday_conditions": "Sunny",
      "chance_of_rain_pct": 2
    },
    {
      "date": "2026-08-22",
      "max_c": 25,
      "min_c": 17,
      "midday_conditions": "Sunny",
      "chance_of_rain_pct": 1
    },
    {
      "date": "2026-08-23",
      "max_c": 24,
      "min_c": 18,
      "midday_conditions": "Overcast ",
      "chance_of_rain_pct": 15
    }
  ]
}


Your app sent no URL, no API key and no query string — only the city. That is the point of
an `http` executor.

There is nothing to write for these two tools. No function, no dispatcher, no code at all.

---

## Step 7 — Write the system prompt

The tools are ready. Now the agent needs to be told when to use them.

Put that in the prompt, not in your code. Then you can change how the agent behaves by
committing a new prompt version, with no deploy and no code review.

Three things in the text below do the real work:

**One rule per tool**, naming the tool and the condition. "Call `convert_currency` only
when the traveller names an amount and two currencies" is far more reliable than hoping
the description alone carries it.

**A rule for using no tool at all.** Without a line like "answer directly, with no tool,
for visas, culture, packing, safety or itineraries", a model that has tools tends to reach
for one anyway.

**The limit of a tool, in words.** The forecast only reaches three days out, so the prompt
tells the model to answer seasonal questions from its own knowledge. That single sentence
stops the agent calling a three-day forecast to answer "when should I visit Japan".

`{{ today }}` is a template variable. The agent needs it because `search_flights` wants a
real date and travellers say "next Friday". Your code fills it in when it renders the
prompt.

### In the dashboard

**Prompts → New prompt**, then one SYSTEM message in the editor and **Commit new version**.

| Field | What to enter |
|---|---|
| **Name** | `travel-planner` |
| **Description** | Travel planner agent. |
| **Message** | one message, role **system** — paste the `SYSTEM_PROMPT` text from the next cell, everything between the triple quotes, `{{ today }}` included |
| **Default model** | `gpt-4o-mini`, or whatever your **Gateway → Models** page calls the model you connected |

Leave the message list at that one system message. No user message, no example turn — the
traveller's question arrives from your code at run time.

![The travel-planner prompt in the editor: one SYSTEM message with the today variable highlighted, default model gpt-4o-mini, production and staging both at v1](../../../../apps/docs/static/img/tutorials/build-a-travel-planner-agent/04-system-prompt.png)

### The same thing in code

**Setup.**

In [10]:
SYSTEM_PROMPT = """You are a travel planning assistant for a European travel agency.

Today's date is {{ today }}. Use it to resolve any relative date the traveller
mentions, such as "next Friday" or "in three weeks".

You have three tools for facts you cannot know on your own: flight availability,
the current weather, and today's exchange rates. Use a tool only when the
traveller's question actually depends on that live data.

How to choose:

- Call search_flights only when the traveller gives a departure city, a
  destination, and a date you can resolve to a single day.
- Call get_city_weather when the answer depends on the weather in the next three
  days. If the question is about a season or a typical climate, answer from your
  own knowledge instead: the forecast only reaches three days out.
- Call convert_currency only when the traveller names an amount and two
  currencies.
- Answer directly, with no tool at all, for visas, culture, packing, safety,
  itineraries, or the best time of year to visit a place.

Never invent a flight number, a price, or a temperature. When a tool returns a
figure, repeat it exactly as given. Keep answers short and practical, and always
include units and currency codes."""

found = [p for p in (await hub.prompts.list(search=PROMPT)).data if p.name == PROMPT]
if found:
    prompt = found[0]
    print("prompt already exists:", prompt.name)
else:
    prompt = await hub.prompts.create(PROMPT, description="Travel planner agent.")
    version = await hub.prompts.commit_version(
        prompt.id,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}],
        model=MODEL,        # so the calling code never hardcodes a model
    )
    print(f"created the prompt and committed version {version.version_number}")

print("prompt id:", prompt.id)

prompt already exists: travel-planner
prompt id: 4cd6fc3e-c4af-48bf-8fa5-0b3081cdc62d


---

## Step 8 — Connect the tools to the prompt

The tools exist and the prompt exists, but they do not know about each other yet. A
**binding** is the link. Binding a tool to the prompt is what puts it in front of the model
on every call — and it is why your calling code never has to name a tool.

Each binding chooses **how** to find the tool version:

- **follow an alias** (`tool_alias="production"`) — the normal choice. Ship a fix by moving
  the tool's `production` alias, and every prompt bound to it picks the fix up. No
  redeploy, no prompt edit.
- **pin one version** (`pinned_version_number=1`) — for a prompt that must keep running
  one exact build, whatever else changes.

### In the dashboard

The prompt's **Tools** tab, then **Connect a tool from the catalog** — once for
`search_flights`, once for `get_city_weather`, once for `convert_currency`. In the
**default** column leave each one following the `production` alias.

Nothing to type here, and nothing to commit: the tab says "Changes save straight away".

![The prompt's Tools tab showing convert_currency, get_city_weather and search_flights each bound in the default column, with two aliases inheriting](../../../../apps/docs/static/img/tutorials/build-a-travel-planner-agent/05-three-tool-bindings.png)

### The same thing in code

**Setup.**

In [11]:
for tool in (flights_tool, weather_tool, currency_tool):
    binding = await hub.prompts.set_tool_binding(prompt.id, tool.id, tool_alias="production")
    print(
        f"{binding.tool_name:18} follows alias {binding.tool_alias!r}"
        f" -> version {binding.resolved_version_number} today"
    )

search_flights     follows alias 'production' -> version 2 today
get_city_weather   follows alias 'production' -> version 2 today
convert_currency   follows alias 'production' -> version 2 today


`set_tool_binding` replaces the binding for that tool instead of adding a second one, so
running this cell again changes nothing.

---

## Step 9 — Run the agent

Two calls. First **render** the prompt: that returns the system message with `{{ today }}`
filled in, the model the version binds, and the tools bound to it. Then run the loop.

**Check.** `render` itself is app code — the next cell uses it for real. The extra `resolve`
loop below is only here to show you which of the three tools your app has to run. Look at
what came back before running anything.

In [12]:
from datetime import date

rendered = await hub.prompts.render(PROMPT, "production", {"today": date.today().isoformat()})

print("model bound to the version:", rendered.model)
print("prompt version:", rendered.version_number)
print()
print("tools the model will be shown:")
refs = [{"name": t["function"]["name"], "alias": "production"} for t in rendered.tools]
for item in await hub.tools.resolve(refs):
    name = item.function["name"]
    who = "your code must run it" if item.executor_type == "client" else "the platform runs it"
    print(f"  {name:18} v{item.version_number}  executor={item.executor_type:7} -> {who}")

model bound to the version: gpt-4o-mini
prompt version: 1

tools the model will be shown:
  search_flights     v2  executor=client  -> your code must run it
  get_city_weather   v2  executor=http    -> the platform runs it
  convert_currency   v2  executor=http    -> the platform runs it


That last column tells you exactly what your app owes the loop. One tool says "your code
must run it", so the map you pass has exactly one entry:

```python
client_tools = {"search_flights": search_flights}
```

**The key is the catalog tool's name**, spelled exactly as the dashboard shows it. That
string is what the model asks for, so the loop matches on it. The value can be any
function you like — its own name does not matter at all.

The two `http` tools are **absent on purpose**. The platform runs those, so there is
nothing for your app to supply. Adding them here would be wrong, not extra safety.

**Your app.** The next cell is the real thing, and it is short: a map with one entry, then
render, append the question, run the loop. `ask()` only exists so the later questions do not
repeat these five lines.

In [13]:
CLIENT_TOOLS = {FLIGHTS: search_flights}


async def ask(question: str):
    """Render the stored prompt, add the traveller's question, run the loop."""
    rendered = await hub.prompts.render(
        PROMPT, "production", {"today": date.today().isoformat()}
    )

    # The user turn is appended here, client-side. A tool loop owns its message list,
    # because every round appends the model's tool calls and the tools' results.
    messages = [*rendered.messages, {"role": "user", "content": question}]

    result = await hub.gateway.run_prompt_with_tools(
        rendered,
        messages=messages,
        client_tools=CLIENT_TOOLS,
        trace={"name": "notebook-travel-planner"},
    )

    called = [
        call["function"]["name"]
        for message in result.messages
        if message.get("role") == "assistant"
        for call in (message.get("tool_calls") or [])
    ]
    print(f"Q: {question}")
    print(f"   rounds: {result.iterations}   tools called: {called or 'none'}")
    print(f"A: {result.content}")
    return result


flight_run = await ask("Any flights from Amsterdam to Lisbon on 2026-08-28?")
print("\ntrace:", flight_run.trace_id)

  -> search_flights(origin='Amsterdam', destination='Lisbon', departure_date='2026-08-28')
Q: Any flights from Amsterdam to Lisbon on 2026-08-28?
   rounds: 2   tools called: ['search_flights']
A: There are three flights available from Amsterdam to Lisbon on August 28, 2026:

1. **KLM**  
   - Flight No: KL1693  
   - Departure: 07:20  
   - Arrival: 09:45  
   - Price: €189  

2. **TAP Air Portugal**  
   - Flight No: TP671  
   - Departure: 12:05  
   - Arrival: 14:30  
   - Price: €154  

3. **Transavia**  
   - Flight No: HV5171  
   - Departure: 18:40  
   - Arrival: 21:05  
   - Price: €118  

Let me know if you need help with anything else!

trace: dc9e9e9f-4de5-4148-b778-e3ed6588d640


Read the output from the bottom up. Your `search_flights` printed its arguments, so you
know the model chose the tool and filled in the fields itself. `rounds: 2` means two model
calls: one to ask for the tool, one to turn the tool's result into an answer.

Nothing in the code above named a tool. The prompt's bindings did.

---

## Step 10 — Watch it choose

**Your app**, three more times. Same prompt, same three tools, and each question needs a
different decision.

In [14]:
no_tool_run = await ask(
    "What's the best time of year to visit Japan, and do I need a visa as a Dutch citizen?"
)
print()
one_http_run = await ask("Should I pack a raincoat for Lisbon? I land tomorrow.")
print()
two_tool_run = await ask(
    "I'm in Lisbon for the next three days with a budget of 500 EUR. "
    "What's the weather, and what is that worth in Japanese yen?"
)

Q: What's the best time of year to visit Japan, and do I need a visa as a Dutch citizen?
   rounds: 1   tools called: none
A: The best times to visit Japan are typically during the spring (March to May) and autumn (September to November) when the weather is mild and the scenery is beautiful, thanks to cherry blossoms in spring and vibrant autumn leaves.

As for visa requirements, Dutch citizens can enter Japan for short stays (up to 90 days) without a visa for tourism, business, or visiting friends and family. However, make sure to check for any updates or additional requirements before your trip.

Q: Should I pack a raincoat for Lisbon? I land tomorrow.
   rounds: 2   tools called: ['get_city_weather']
A: Lisbon has sunny weather with a very low chance of rain (2%) tomorrow. You likely won't need a raincoat. However, just to be cautious, it might be a good idea to pack a light jacket for the evening when temperatures can be cooler. Safe travels!

Q: I'm in Lisbon for the next three da

Four questions, four different decisions:

| Question | Tools called |
|---|---|
| best time to visit Japan, visa | none |
| flights Amsterdam to Lisbon | `search_flights` |
| raincoat for Lisbon tomorrow | `get_city_weather` |
| weather in Lisbon plus 500 EUR in yen | `get_city_weather` **and** `convert_currency` |

The last one is the interesting case. The model asked for **two** tools in one turn, and
the loop ran both at the same time before sending both results back together. In the trace
their bars overlap, nested under the round that asked for them:

![Trace named runToolLoop with 4 spans: one LLM span containing convert_currency and get_city_weather, then a second LLM span](../../../../apps/docs/static/img/tutorials/build-a-travel-planner-agent/06-trace-two-tools.png)

The `client` tool looks the same in a trace, except its bar has no measurable length — it
ran inside your own process, not over the network:

![Trace with 3 spans showing search_flights at 0 ms nested under the first LLM span](../../../../apps/docs/static/img/tutorials/build-a-travel-planner-agent/08-trace-client-tool.png)

And the Japan question is one single span, because the loop ended on round one:

![Trace named runToolLoop with 1 span: one LLM span with nothing nested under it](../../../../apps/docs/static/img/tutorials/build-a-travel-planner-agent/07-trace-no-tool.png)

**Check.** Rather than trust those pictures, read the traces back from the API. Traces are sent in
the background so they take a second or two to arrive.

Expect your own numbers to differ from the screenshots by a few percent. The screenshots
come from one real run, and a model's answer is a little longer or shorter every time, so
the token counts move with it. What stays the same is the shape: one span for no tool,
three for one tool, four for two tools.

In [15]:
import asyncio


def walk(spans):
    """Flatten the parent/child span tree."""
    for span in spans:
        yield span
        yield from walk(span.children)


async def summarise(trace_id, attempts=15):
    for _ in range(attempts):
        detail = await hub.traces.get(trace_id)
        spans = list(walk(detail.spans))
        if spans:
            return detail.trace, spans
        await asyncio.sleep(1)
    return None, []


rows = [
    ("no tool", no_tool_run),
    ("one client tool", flight_run),
    ("one http tool", one_http_run),
    ("two http tools", two_tool_run),
]
print(f"{'run':18} {'spans':>5} {'tokens':>7} {'cost usd':>10}  tools")
for label, run in rows:
    trace, spans = await summarise(run.trace_id)
    tools = [s.name for s in spans if s.kind == "tool"]
    print(f"{label:18} {trace.span_count:>5} {trace.total_tokens:>7} {trace.total_cost_usd:>10.6f}  {tools or '-'}")

run                spans  tokens   cost usd  tools
no tool                1     645   0.000140  -
one client tool        3    1471   0.000303  ['search_flights']
one http tool          3    1358   0.000236  ['get_city_weather']
two http tools         4    1574   0.000318  ['convert_currency', 'get_city_weather']


Look at the tokens column. A tool call roughly doubles them, because the tool definitions,
the model's call and the tool's result all become input on the next round.

So answering with **no tool is not a weaker outcome** — it is the cheaper and faster one,
and getting it right is most of what separates a useful agent from an annoying one. If your
agent reaches for a tool too often, the fix is in the system prompt and the tool
descriptions. Both are versioned, and neither needs a deploy.

---

## Step 11 — Four ways to get this wrong

Every one of these is run for real below, so you can read the actual error once and
recognise it later. All four cells are **broken on purpose** — none of them is app code.

### Mistake 1 — a shell with no version

The tool exists, the dashboard lists it, and it still cannot be used. This is step 1's
warning, as an error message.

In [16]:
from acruxcore.errors import AcruxCoreError

orphan = await hub.tools.create("tp_shell_with_no_version", description="A shell, on purpose.")
print("created:", orphan.name)

try:
    await hub.tools.resolve([{"name": orphan.name, "alias": "production"}])
except AcruxCoreError as exc:
    print("code:", exc.code)
    print(exc)
finally:
    await hub.tools.delete(orphan.id)
    print("(deleted the throwaway tool again)")

created: tp_shell_with_no_version
code: API_ERROR
acruxcore API error 404 resolving tools
(deleted the throwaway tool again)


A plain `404`. The tool is really there, but it has no `production` alias to follow,
because aliases are only minted by a first version — and there is no first version. If a
brand new tool "cannot be found", count its versions before anything else.

### Mistake 2 — a `client` tool with no implementation

**Broken on purpose.** Here the key in `client_tools` is misspelled, so nothing can run the
tool. This one is
friendly: it is raised **before** the first model call, so you are not billed, and the
message prints the keys you did pass so you can see the typo.

In [17]:
try:
    await hub.gateway.run_prompt_with_tools(
        rendered,
        messages=[*rendered.messages, {"role": "user", "content": "Flights to Lisbon?"}],
        client_tools={"search_flght": search_flights},        # deliberate typo
        trace=False,
    )
except AcruxCoreError as exc:
    print("code:", exc.code)
    print(exc)

code: MISSING_DISPATCH
acruxcore: tool 'search_flights' has a client executor, so something has to run it, but no implementation was supplied. Pass it in client_tools={'search_flights': ...}, or pass dispatch=. client_tools held: ['search_flght'].


### Mistake 3 — a function that cannot take the schema's arguments

**Broken on purpose.** The key is right this time, but the function takes a single `query`
argument while the schema sends `origin`, `destination` and `departure_date`. This is also caught before the
first model call.

In [18]:
def wrong_signature(query: str) -> dict:
    """Looks reasonable, but the schema does not send a `query` field."""
    return {"flights": []}


try:
    await hub.gateway.run_prompt_with_tools(
        rendered,
        messages=[*rendered.messages, {"role": "user", "content": "Flights to Lisbon?"}],
        client_tools={FLIGHTS: wrong_signature},
        trace=False,
    )
except AcruxCoreError as exc:
    print("code:", exc.code)
    print(exc)

code: VALIDATION_ERROR
acruxcore: the function passed in client_tools for 'search_flights' cannot receive this tool's arguments — the catalog schema requires ['origin', 'destination', 'departure_date'] and the function accepts ['query']. A client_tools function is called with the schema's own parameter names, so define it as search_flights(origin, destination, departure_date), or accept **kwargs.


### Mistake 4 — no tool reaches the model, and nothing complains

This is the dangerous one, because there is **no error at all**. If no tool is bound to the
prompt, `render` returns an empty tool list, the call becomes a plain completion, and the
model answers from its own knowledge. Your function is never called and your logs look
fine.

**Broken on purpose.** `tool_refs=[]` below reproduces exactly what an unbound prompt does.

In [19]:
silent = await hub.gateway.run_prompt_with_tools(
    rendered,
    messages=[
        *rendered.messages,
        {"role": "user", "content": "Any flights from Amsterdam to Lisbon on 2026-08-28?"},
    ],
    tool_refs=[],                 # same effect as a prompt with nothing bound
    client_tools=CLIENT_TOOLS,    # supplied, and never used
    trace=False,
)
print(silent.content)
print("\n^ no error, and no tool call - your function was never reached")

Please hold on while I check the flight availability from Amsterdam to Lisbon on August 28, 2026.

^ no error, and no tool call - your function was never reached


Read that answer carefully. Sometimes the model invents a flight number, and sometimes, as
here, it promises to go and look and then stops. Both are bad, and neither raises anything
you could catch.

So when a tool "does nothing", check the prompt's **Tools** tab before you start debugging
your Python.

---

## Step 12 — Close the client

**Your app.** Traces are reported in the background, so close the client when you finish. That flushes
whatever is still waiting to be sent. In a script `async with AcruxCore() as hub:` does it
for you. A notebook has no block to leave, so call it yourself.

In [20]:
await hub.gateway.aclose()
print("flushed")

flushed


---

## What you built

- Three tool definitions in the catalog: one `client`, two `http`.
- One prompt version holding all the routing rules, editable without a deploy.
- Three bindings connecting them.
- An agent whose calling code passes a question and one function, and nothing else.

Everything that decides the agent's behaviour — the tool descriptions, the schemas, the
system prompt — is versioned on the platform. Your code carries the flight inventory and
nothing more.

### What of this actually goes into your app

Almost none of it. Setup happens once — by clicking, or with the create-and-commit calls —
and the checks were for you, not for the program. What ships is this:

```python
# 1. the implementation of the one client tool
def search_flights(origin: str, destination: str, departure_date: str) -> dict:
    ...                                    # your inventory lookup

CLIENT_TOOLS = {"search_flights": search_flights}

# 2. render the stored prompt, add the question, run the loop
rendered = await hub.prompts.render(
    "travel-planner", "production", {"today": date.today().isoformat()}
)
result = await hub.gateway.run_prompt_with_tools(
    rendered,
    messages=[*rendered.messages, {"role": "user", "content": question}],
    client_tools=CLIENT_TOOLS,
)
print(result.content)
```

That is the whole agent. No tool schema, no tool description, no model name, no list of
which tools exist — all of that came from the platform through `rendered`.

Everything else you ran here is scaffolding: `find_tool` and `create_tool_if_missing` are
this notebook's own helpers so it can be re-run safely, and the preflight cell,
`tools.resolve`, `tools.execute` and the trace summary are checks. None of them is an SDK
requirement, and none of them belongs in your app.

### What this notebook left in your team

One prompt (`travel-planner`) and three tools (`search_flights`, `get_city_weather`,
`convert_currency`). Every cell is find-or-create, so running the notebook again is safe.
Delete them from the dashboard when you are done, or keep them — the tutorial's
screenshots match these exact objects.

### Where to go next

- [Call a prompt's tools from the SDK](https://docs.acruxcore.com/docs/guides/call-a-prompts-tools-from-the-sdk)
  — every shape this loop can take, including streaming the rounds as they arrive.
- [Define a tool in code, or in the catalog](https://docs.acruxcore.com/docs/guides/define-a-tool-in-code-or-in-the-catalog)
  — let a decorated Python function own the definition instead of the catalog.
- [Manage a tool's lifecycle via the SDK](https://docs.acruxcore.com/docs/guides/manage-a-tools-lifecycle-via-the-sdk)
  — versions, aliases and `http` executors without a dashboard click.